# Part II · Lesson 8 — Approve or reject a refund

### Import the graph helpers

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

### Describe the refund request

In [ ]:
class RefundState(TypedDict):
    order_id: str
    amount: int
    reason: str
    approved: bool
    status: str

### Prepare the request for review

In [ ]:
def prepare_refund(state: RefundState) -> dict:
    if state["amount"] <= 0:
        raise ValueError("The refund amount must be positive.")
    return {"status": "awaiting review"}

### Pause and request an explicit decision

In [ ]:
def review_refund(state: RefundState) -> dict:
    decision = interrupt({
        "order_id": state["order_id"], "amount_inr": state["amount"],
        "reason": state["reason"], "question": "Approve this refund?",
    })
    if not isinstance(decision, bool):
        raise ValueError("Resume with True or False.")
    return {"approved": decision, "status": "reviewed"}

### Define approval and rejection outcomes

In [ ]:
def process_refund(state: RefundState) -> dict:
    # Demo only: record the result without moving money.
    return {"status": "refund processed"}


def reject_refund(state: RefundState) -> dict:
    return {"status": "refund rejected"}

### Choose the branch after review

In [ ]:
def refund_route(state: RefundState) -> str:
    return "approve" if state["approved"] else "reject"

### Register and connect the refund workflow

In [ ]:
builder = StateGraph(RefundState)
builder.add_node("prepare", prepare_refund)
builder.add_node("review", review_refund)
builder.add_node("process", process_refund)
builder.add_node("reject", reject_refund)
builder.add_edge(START, "prepare")
builder.add_edge("prepare", "review")
builder.add_conditional_edges(
    "review", refund_route, {"approve": "process", "reject": "reject"}
)
builder.add_edge("process", END)
builder.add_edge("reject", END)

### Attach memory and prepare the sample request

In [ ]:
memory = InMemorySaver()
graph = builder.compile(checkpointer=memory)
refund_input = {
    "order_id": "ORD-102", "amount": 5000, "reason": "Item arrived damaged.",
    "approved": False, "status": "new",
}
approve_config = {"configurable": {"thread_id": "refund-review-1"}}

### Start the workflow and inspect the pause

In [ ]:
paused = graph.invoke(refund_input, approve_config)
print(paused["__interrupt__"][0].value)
print(graph.get_state(approve_config).values["status"])

### Resume with approval

In [ ]:
approved_response = graph.invoke(Command(resume=True), approve_config)
print(approved_response["approved"], approved_response["status"])

### Try rejection in another review

In [ ]:
reject_config = {"configurable": {"thread_id": "refund-review-2"}}
graph.invoke(refund_input, reject_config)
rejected_response = graph.invoke(Command(resume=False), reject_config)
print(rejected_response["approved"], rejected_response["status"])